<a href="https://colab.research.google.com/github/GD-0305/Assignment-1---Financial-Econometrics/blob/main/Assignment_1_Financial_Econometrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Michael Tinashe Goredema - R2420839

# Financial Econometrics — Project 1
## Best-Practices Handbook: Challenges in Time Series Modeling

This notebook covers four challenges that come up when modeling financial time series:
1. Multicollinearity
2. Skewness
3. Sensitivity to Outliers
4. Overfitting

We use daily stock return data for AAPL (primary data), MSFT, GOOG, and NVDA downloaded from Yahoo Finance.

In [ ]:
# install packages
!pip install yfinance statsmodels scikit-learn numpy pandas matplotlib seaborn scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print('libraries loaded')

In [ ]:
# download daily closing prices for 4 tech stocks
tickers = ['AAPL', 'MSFT', 'GOOG', 'NVDA']
prices = yf.download(tickers, start='2018-01-01', end='2025-12-31', auto_adjust=True)['Close']

# compute daily log returns
returns = np.log(prices / prices.shift(1)).dropna()
returns.columns = tickers

print('shape:', returns.shape)
returns.head()

---
## Challenge 1: Multicollinearity

### Definition

Multicollinearity happens when two or more predictor variables in a regression are highly correlated with each other. When this occurs, the matrix $\mathbf{X}^\top \mathbf{X}$ becomes close to singular and the OLS estimates become unreliable.

We measure it using the **Variance Inflation Factor (VIF)**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ comes from regressing $X_j$ on all other predictors. A VIF above 10 is generally considered a problem.

### Description

In simple terms, multicollinearity means some of the predictors are carrying the same information. The model struggles to figure out which variable is actually driving the outcome, so the coefficients become unstable and hard to interpret.

In [ ]:
# Demonstration
# regress AAPL returns on the other three stocks

X = returns[['MSFT', 'GOOG', 'NVDA']]
y = returns['AAPL']

X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:
# calculate VIF for each predictor
vif_df = pd.DataFrame()
vif_df['variable'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print('VIF Results:')
print(vif_df)
print('')
print('VIF > 10 means severe multicollinearity')

In [ ]:
# Diagram: correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(returns.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Stock Returns')
plt.tight_layout()
plt.savefig('multicollinearity_heatmap.png', dpi=120)
plt.show()

### Diagnosis

There are a few ways to check for multicollinearity:
- Look at the pairwise correlation matrix. Correlations above 0.8 between predictors are a warning sign.
- Calculate VIF values. VIF > 5 is moderate, VIF > 10 is severe.
- Check if regression coefficients change a lot when you add or remove variables — instability is a symptom.

### Damage

When multicollinearity is present, the standard errors of the regression coefficients get inflated. This means you might get a high $R^2$ but the individual coefficients are not statistically significant and can even have the wrong sign. For a derivatives desk, this is dangerous because it means the model cannot reliably identify which factors are actually driving volatility.

### Directions

Some ways to deal with multicollinearity:
1. **Ridge regression** — adds a penalty to shrink the coefficients, which stabilises them even when predictors are correlated.
2. **Remove one of the correlated variables** — if two variables are measuring almost the same thing, just keep one.
3. **Principal Component Analysis (PCA)** — transforms the predictors into uncorrelated components before running the regression.
4. **Increasing sample size** - the logic in this particular procedure lies in the model's ability to yield estimates that converge to their true values as the sample size increases to infinity.

In [ ]:
# compare OLS vs Ridge coefficients
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ols = LinearRegression().fit(X_scaled, y)
ridge = Ridge(alpha=1.0).fit(X_scaled, y)

coef_compare = pd.DataFrame({
    'variable': tickers[1:],
    'OLS coef': ols.coef_.round(5),
    'Ridge coef': ridge.coef_.round(5)
})
print(coef_compare)
print('')
print('Ridge shrinks the coefficients slightly, reducing the impact of collinearity')

---
## Challenge 2: Skewness

### Definition

Skewness measures how asymmetric a distribution is around its mean. The formula for skewness is:

$$\text{Skewness} = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^3}{\left(\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2\right)^{3/2}}$$

A value of 0 means symmetric. Negative skewness means the left tail is longer (more extreme negative returns than positive). Positive skewness means the right tail is longer.

### Description

Financial returns tend to be negatively skewed — big crashes happen more often than big rallies of the same size. If we assume returns are normally distributed (symmetric), we end up underestimating the probability of large losses, which is a serious problem for risk management.

In [ ]:
# Demonstration: check skewness of each stock
print('Skewness and Kurtosis of Daily Log Returns')
print('='*45)
for ticker in tickers:
    sk = returns[ticker].skew()
    ku = returns[ticker].kurtosis()
    print(f'{ticker}:  skewness = {sk:.4f},  excess kurtosis = {ku:.4f}')

print('')
print('Negative skewness confirms left-tail asymmetry in all stocks')

In [ ]:
# Jarque-Bera test for normality
print('Jarque-Bera Test (tests if data is normally distributed)')
print('='*55)
for ticker in tickers:
    jb_stat, p_val = stats.jarque_bera(returns[ticker])
    result = 'reject normality' if p_val < 0.05 else 'cannot reject normality'
    print(f'{ticker}:  JB stat = {jb_stat:.2f},  p-value = {p_val:.4f}  => {result}')

In [ ]:
# Diagram: histogram of AAPL returns vs normal distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# histogram
r = returns['AAPL']
axes[0].hist(r, bins=60, density=True, color='steelblue', alpha=0.6, label='AAPL returns')
x = np.linspace(r.min(), r.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'r-', linewidth=2, label='Normal distribution')
axes[0].set_title('AAPL Log Returns vs Normal Distribution')
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Density')
axes[0].legend()

# QQ plot
stats.probplot(r, dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot — AAPL Returns')

plt.tight_layout()
plt.savefig('skewness_plots.png', dpi=120)
plt.show()
print('The histogram shows the left tail is heavier than the normal distribution predicts')

### Diagnosis

- Calculate the sample skewness. Values below -0.5 or above 0.5 indicate meaningful asymmetry.
- Run the Jarque-Bera test. A small p-value means the data is not normally distributed.
- Look at a histogram and compare it to a normal curve — you can often see the asymmetry visually.
- A QQ plot will show deviations from the straight line, especially in the tails.

### Damage

If a model assumes symmetry but the data is negatively skewed, it will underestimate how often large negative returns occur. For options pricing, this means put options (which pay off during market crashes) will be underpriced. For risk management, Value-at-Risk calculations based on the normal distribution will understate the true downside risk.

### Directions

1. **Use a skewed distribution** — fit a skewed-t or a GED (Generalised Error Distribution) instead of assuming normality.
2. **Transform the data** — a Box-Cox or log transformation can reduce skewness in some cases.
3. **Use GARCH with non-normal errors** — for example, a GARCH model with Student-t innovations handles fat tails and some asymmetry better than Gaussian GARCH.

---
## Challenge 3: Sensitivity to Outliers

### Definition

A model is sensitive to outliers when a small number of extreme observations have a large effect on the estimated parameters. In OLS regression, this is measured using **Cook's Distance**:

$$D_i = \frac{\sum_{j=1}^{n}(\hat{y}_j - \hat{y}_{j(-i)})^2}{p \cdot MSE}$$

where $\hat{y}_{j(-i)}$ is the fitted value when observation $i$ is removed. A common rule of thumb is that $D_i > 4/n$ flags an influential observation.

### Description

Outliers in financial data often come from major market events — like the COVID crash in March 2020 or the 2022 rate hike shock. OLS tries to fit every point equally, so these extreme days can pull the regression line significantly away from where it should be.

In [ ]:
# Demonstration: regress AAPL on GOOG and find influential observations
X_simple = sm.add_constant(returns[['GOOG']])
y_simple = returns['AAPL']

model_ols = sm.OLS(y_simple, X_simple).fit()

# get Cook's distance
influence = model_ols.get_influence()
cooks_d = influence.cooks_distance[0]
std_resid = influence.resid_studentized_internal

n = len(y_simple)
threshold = 4 / n

n_outliers = (np.abs(std_resid) > 3).sum()
n_influential = (cooks_d > threshold).sum()

print(f'Total observations: {n}')
print(f'Outliers (|std residual| > 3): {n_outliers}')
print(f'Influential points (Cooks D > 4/n): {n_influential}')

# show the most extreme days
top5 = np.argsort(cooks_d)[-5:][::-1]
print('')
print('Top 5 most influential trading days:')
for i in top5:
    print(f"  {returns.index[i].date()}  AAPL={returns['AAPL'].iloc[i]:.4f}  GOOG={returns['GOOG'].iloc[i]:.4f}  Cook's D={cooks_d[i]:.6f}")

In [ ]:
# Diagram: show how outliers shift the regression line
influential_mask = cooks_d > threshold

# fit model without influential points
X_clean = sm.add_constant(returns['GOOG'][~influential_mask])
model_clean = sm.OLS(returns['AAPL'][~influential_mask], X_clean).fit()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# scatter with regression lines
x_range = np.linspace(returns['GOOG'].min(), returns['GOOG'].max(), 200)
axes[0].scatter(returns['GOOG'][~influential_mask], returns['AAPL'][~influential_mask],
               alpha=0.2, s=5, color='steelblue', label='normal obs')
axes[0].scatter(returns['GOOG'][influential_mask], returns['AAPL'][influential_mask],
               alpha=0.8, s=20, color='red', label='influential obs')
axes[0].plot(x_range, model_ols.params[0]   + model_ols.params[1]   * x_range, 'r-',  lw=2, label='OLS full data')
axes[0].plot(x_range, model_clean.params[0] + model_clean.params[1] * x_range, 'g--', lw=2, label='OLS without outliers')
axes[0].set_title('Effect of Outliers on Regression Line')
axes[0].set_xlabel('GOOG return')
axes[0].set_ylabel('AAPL return')
axes[0].legend(fontsize=8)

# Cook's distance plot
axes[1].bar(range(len(cooks_d)), cooks_d, color='steelblue', alpha=0.6)
axes[1].axhline(threshold, color='red', linestyle='--', label=f'threshold = 4/n')
axes[1].set_title("Cook's Distance")
axes[1].set_xlabel('Observation')
axes[1].set_ylabel("Cook's D")
axes[1].legend()

plt.tight_layout()
plt.savefig('outlier_plots.png', dpi=120)
plt.show()

### Diagnosis

- Check standardised residuals — values beyond ±3 suggest outliers.
- Compute Cook's Distance for each observation. Values above $4/n$ are worth investigating.
- Plot residuals over time and look for spikes on specific dates that correspond to known market events.

### Damage

When a few extreme observations dominate the regression, the estimated coefficients shift away from the true underlying relationship. This means volatility estimates and beta values become unreliable. In practice, a model calibrated on data that includes a market crash will produce inflated volatility forecasts during calm periods — making derivative prices too expensive and potentially costing the desk business.

### Directions

1. **Winsorisation** — cap extreme returns at the 1st and 99th percentiles before fitting the model. Simple and effective.
2. **Robust regression** — uses a loss function that is less sensitive to large residuals, such as Huber regression.
3. **Investigate and remove known bad data** — sometimes outliers are data errors, not real market events, and should simply be excluded.

---
## Challenge 4: Overfitting

### Definition

Overfitting happens when a model is too complex and ends up learning the noise in the training data rather than the true underlying pattern. The result is a model that performs well on the data it was trained on but poorly on new, unseen data.

This is captured by the **bias-variance tradeoff**:

$$\text{Expected MSE} = \text{Bias}^2 + \text{Variance} + \sigma^2_{\varepsilon}$$

A simple model has high bias but low variance. A very complex model has low bias but high variance. Overfitting corresponds to high variance.

The adjusted $R^2$ penalises for adding extra variables:

$$\bar{R}^2 = 1 - \frac{(1 - R^2)(n-1)}{n - p - 1}$$

### Description

In financial modeling, overfitting often happens when someone fits a very flexible model to historical return data and gets a great in-sample fit, only to find that the model fails completely on future data. This is sometimes called "curve fitting" and is a common trap in quantitative finance.

In [ ]:
# Demonstration: fit polynomial models of increasing complexity
# use MSFT returns to predict AAPL returns

X_data = returns['GOOG'].values.reshape(-1, 1)
y_data = returns['AAPL'].values

# split into train (70%) and test (30%) — keep time order
split = int(len(X_data) * 0.7)
X_train, X_test = X_data[:split], X_data[split:]
y_train, y_test = y_data[:split], y_data[split:]

print('Train/test split done')
print(f'Training observations: {len(X_train)}')
print(f'Test observations:     {len(X_test)}')
print('')

# fit polynomial models of degree 1, 2, 4, 8
print(f'{"Degree":<10}{"Train RMSE":<15}{"Test RMSE":<15}')
print('-'*38)
for degree in [1, 2, 4, 8]:
    poly = PolynomialFeatures(degree=degree)
    scaler = StandardScaler()

    Xtr_poly = scaler.fit_transform(poly.fit_transform(X_train))
    Xte_poly = scaler.transform(poly.transform(X_test))

    reg = LinearRegression().fit(Xtr_poly, y_train)

    train_rmse = np.sqrt(mean_squared_error(y_train, reg.predict(Xtr_poly)))
    test_rmse  = np.sqrt(mean_squared_error(y_test,  reg.predict(Xte_poly)))

    print(f'{degree:<10}{train_rmse:<15.6f}{test_rmse:<15.6f}')

print('')
print('Train RMSE keeps falling but test RMSE rises — classic overfitting')

In [ ]:
# Diagram: train vs test error by complexity
degrees = [1, 2, 3, 4, 5, 6, 7, 8]
train_errors = []
test_errors  = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(poly.fit_transform(X_train))
    Xte = scaler.transform(poly.transform(X_test))
    reg = LinearRegression().fit(Xtr, y_train)
    train_errors.append(np.sqrt(mean_squared_error(y_train, reg.predict(Xtr))))
    test_errors.append(np.sqrt(mean_squared_error(y_test,  reg.predict(Xte))))

plt.figure(figsize=(9, 5))
plt.plot(degrees, train_errors, 'g-o', label='Train RMSE')
plt.plot(degrees, test_errors,  'r-s', label='Test RMSE')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('RMSE')
plt.title('Overfitting: Train vs Test Error as Complexity Increases')
plt.legend()
plt.tight_layout()
plt.savefig('overfitting_curve.png', dpi=120)
plt.show()

### Diagnosis

- Split the data into training and test sets. If train error is much lower than test error, the model is overfitting.
- Use cross-validation to get a more reliable estimate of out-of-sample error.
- Compare adjusted $R^2$ — if it decreases when you add a variable, that variable is not helping.
- Use AIC or BIC to compare models — lower values indicate a better trade-off between fit and complexity.

### Damage

An overfit model gives a false sense of accuracy. It will perform well on historical data but fail when used for actual forecasting. For a derivatives desk, a model that looks great in backtesting but fails in live trading is one of the most costly mistakes — derivative pricing errors translate directly into financial losses.

### Directions

1. **Use simpler models** — GARCH(1,1) often outperforms more complex volatility models out of sample. Adding complexity is not always better.
2. **Ridge or Lasso regularisation** — these penalise large coefficients, which prevents the model from fitting the noise.
3. **Cross-validation** — always evaluate model performance on data that was not used for training before deploying the model.

In [ ]:
# Ridge vs OLS for the overfitting case (degree 8)
from sklearn.linear_model import Ridge

poly = PolynomialFeatures(degree=8)
scaler = StandardScaler()
Xtr = scaler.fit_transform(poly.fit_transform(X_train))
Xte = scaler.transform(poly.transform(X_test))

ols_overfit = LinearRegression().fit(Xtr, y_train)
ridge_model = Ridge(alpha=1.0).fit(Xtr, y_train)

print('Degree-8 model: OLS vs Ridge')
print(f'OLS   — train RMSE: {np.sqrt(mean_squared_error(y_train, ols_overfit.predict(Xtr))):.6f},  test RMSE: {np.sqrt(mean_squared_error(y_test, ols_overfit.predict(Xte))):.6f}')
print(f'Ridge — train RMSE: {np.sqrt(mean_squared_error(y_train, ridge_model.predict(Xtr))):.6f},  test RMSE: {np.sqrt(mean_squared_error(y_test, ridge_model.predict(Xte))):.6f}')
print('')
print('Ridge reduces test RMSE by preventing the model from overfitting to the training data')

---
## Summary

| Challenge | How to detect | Suggested fix |
|-----------|--------------|---------------|
| Multicollinearity | VIF > 10, high pairwise correlations | Ridge regression, remove one correlated variable |
| Skewness | Sample skewness, Jarque-Bera test, QQ plot | Skewed-t distribution, GARCH with non-normal errors |
| Sensitivity to Outliers | Cook's Distance, standardised residuals | Winsorisation, robust regression |
| Overfitting | Train vs test RMSE gap, adjusted R² | Regularisation, cross-validation, simpler models |

These four challenges are all very common in financial time series. Being aware of them and knowing how to test for them is an important part of building models that actually work in practice.


#Non-Technical Report

##Section 1 - What I Found
This analysis examined the daily stock returns across four major companies and firms, and identified four key problems that, if ignored, could lead to investment problems.
- Stocks move together: AAPL, MSFT, NVDA and GOOG are highly correlated - when one moves, they all tend to move with it. A model using all four stocks may fail to adequately tell which stock is actually driving your portfolio's performance.
- Returns are not symmetric: All four stocks showed more frequent large losses than large gains of the same size.
- Extreme days dominate: A small number of trading days - concentrated around COVID and the 2022 interest rate shock - had outsized influence on every risk estimate.
- Models fit history, not the future: More complex models performed better on historical data, but significantly worse on new data.

##Section 2 - Recommended Course of Action
- Don't take correlated stocks in tech as independent bets: Holding AAPL, MSFT, GOOG and NVDA stocks together does not offer much diversification. Thus, portfolio construction should take this into account, as when one of these stocks falls, the others also tend to fall with.
- Assume losses will be worse than the average suggests: Because returns are negatively skewed, risk limits and hedge costs should be set more conservatively than standard calculations imply.
- Do not let shock periods permanently inflate risk estimates: The COVID crash and 2022 rate shock were extreme, identifiable events. These periods should be isolated and assessed separately, lest they result in protection being overpriced in calm periods and the desk leaving money on the table.
- Prefer simpler, proven models to complex ones: For live trading and risk management, a simple, straightforward and well-validated approach will outperform an elaborate one that was tuned by past data.

##Section 3 - What drove the problems
- Stocks move together: Sector concentration - all the firms are driven by the same macro forces (rate, USD, tech sentiment).
- Asymmetric losses: Since market crashes are sudden and sharp and recoveries are gradual, this structural asymmetry is permanent in equity markets.
- Extreme day influence: COVID-19 pandemic shock and one of the fastest interest rate hiking cycles.
- Models over-fitted to history: The 2019-2021 long bull-run (unusually long and low volatility) made simple trend following seem reliable (deceptively so) and so masked its failure in regime shifts.

In summary, the four stocks studied behave more like a concentrated bet rather than a diversified portfolio.

---
## References

- Belsley, D.A., Kuh, E., & Welsch, R.E. (1980). *Regression Diagnostics*. John Wiley & Sons.
- Hansen, P.R., & Lunde, A. (2005). A Forecast Comparison of Volatility Models: Does Anything Beat a GARCH(1,1)? *Journal of Applied Econometrics*, 20(7), 873–889.
- Hoerl, A.E., & Kennard, R.W. (1970). Ridge Regression: Biased Estimation for Nonorthogonal Problems. *Technometrics*, 12(1), 55–67.
- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer.